In [9]:
from pathlib import Path
import sys

import pandas as pd

HERE = Path.cwd().resolve()

PYTHON_ROOT = next(
    path
    for path in (HERE, *HERE.parents)
    if (path / "trading_portfolio").is_dir()
    and (path / "t212_universe").is_dir()
)

PROJECT_DIR = (
    PYTHON_ROOT
    / "trading_portfolio"
    / "uk_portfolio"
    / "equity_momentum"
)

UNIVERSE_DIR = PYTHON_ROOT / "t212_universe"

source_path = str(PROJECT_DIR / "src")
if source_path not in sys.path:
    sys.path.insert(0, source_path)

BASELINE = {
    "formation_sessions": 252,
    "skip_sessions": 21,
    "top_frac": 0.20,
    "rebalance_frequency": "monthly",
    "initial_capital_gbp": 10_000.0,
}

display(pd.Series(BASELINE, name="Baseline").to_frame())

,Baseline
formation_sessions,252
skip_sessions,21
top_frac,0.2
rebalance_frequency,monthly
initial_capital_gbp,10000.0


In [10]:
import importlib
import momentum_data

importlib.reload(momentum_data)

universe = momentum_data.load_candidate_universe(
    UNIVERSE_DIR,
    PROJECT_DIR / "data" / "candidate_universe_v1.csv",
)

print(f"Frozen share candidates: {len(universe):,}")

display(
    universe[
        ["yf_symbol", "name", "isin", "quote_unit", "exchange"]
    ].head(10)
)

Frozen share candidates: 669


,yf_symbol,name,isin,quote_unit,exchange
0,88E.L,88 Energy,AU00000088E2,GBX,London Stock Exchange AIM
1,AURA.L,Aura Energy,AU000000AEE7,GBX,London Stock Exchange AIM
2,CLA.L,Celsius Resources,AU000000CLA6,GBX,London Stock Exchange AIM
3,EMH.L,European Metals Holdings,AU000000EMH5,GBX,London Stock Exchange AIM
4,GEO.L,Geo Exploration,AU000000GBP6,GBX,London Stock Exchange AIM
5,LIT.L,Litigation Capital Management,AU000000LCA6,GBX,London Stock Exchange AIM
6,SVML.L,Sovereign Metals,AU000000SVM6,GBX,London Stock Exchange AIM
7,WNX.L,Wellnex Life,AU0000162281,GBX,London Stock Exchange AIM
8,SYN.L,Synergia Energy,AU0000233538,GBX,London Stock Exchange AIM
9,ALL.L,Atlantic Lithium,AU0000237554,GBX,London Stock Exchange AIM


In [11]:
import momentum_preparation

importlib.reload(momentum_preparation)

# Builds from existing local caches once; later runs load the frozen snapshot.
prepared_data = momentum_preparation.prepare_momentum_data(PROJECT_DIR)

prices = prepared_data["prices"]
market = prepared_data["market"]
schedule = prepared_data["schedule"]
price_coverage = prepared_data["coverage"]

pd.testing.assert_frame_equal(universe, prepared_data["universe"])
display(pd.Series(prepared_data["manifest"]["summary"], name="Prepared data").to_frame())

ValueError: Preparation inputs changed: src/momentum_preparation.py; review before rebuilding.

In [ ]:
# Availability is separate from trading eligibility, which comes next.
display(
    price_coverage.groupby("source").agg(
        candidates=("yf_symbol", "size"),
        usable_close_sessions=("usable_close_sessions", "sum"),
    )
)

display(
    price_coverage.loc[
        price_coverage["unit_quarantine_sessions"].gt(0),
        ["yf_symbol", "first_usable_close", "last_usable_close", "unit_quarantine_sessions"],
    ].set_index("yf_symbol")
)

,candidates,usable_close_sessions
source,,
additional_yahoo,20,45112
inherited_reconciled,573,1312540
unavailable,76,0


,first_usable_close,last_usable_close,unit_quarantine_sessions
yf_symbol,,,
SPDI.L,2015-01-02,2025-07-30,107
BAY.L,2021-09-30,2025-08-19,93
VOX.L,2022-10-31,2024-02-02,483


In [ ]:
# These dated suspensions were added to the inherited suspension register.
display(
    pd.DataFrame(prepared_data["review"]["suspensions"])[
        ["ticker", "start", "first_suspended_open", "resume"]
    ]
)

# Raw downloads remain unchanged. No replacement prices or fills are invented.
audit_columns = ["invalid_price", "invalid_volume", "suspected_unit_jump"]
display(
    pd.DataFrame({
        "Before review": prepared_data["raw_audit"][audit_columns].sum(),
        "After quarantine": prepared_data["cleaned_audit"][audit_columns].sum(),
    })
)

,ticker,start,first_suspended_open,resume
0,BLU.L,2018-07-26,2018-07-26,2019-01-24
1,SCGL.L,2016-03-29,2016-03-29,2017-02-28
2,SCGL.L,2018-04-30,2018-04-30,2018-10-30
3,SPDI.L,2025-11-06,2025-11-06,None


,Before review,After quarantine
invalid_price,0,0
invalid_volume,0,0
suspected_unit_jump,45,0


In [ ]:
unavailable_candidates = price_coverage.loc[
    ~price_coverage["has_prices"], ["yf_symbol", "name", "unavailable_reason"]
].copy()

print(f"Unavailable candidates retained in the catalogue: {len(unavailable_candidates):,}")
print("No signals, performance results, or development/holdout split have been built yet.")
print("Prices are daily reference observations; liquidity and execution checks still apply.")
display(unavailable_candidates.head(10))

Unavailable candidates retained in the catalogue: 76
No signals, performance results, or development/holdout split have been built yet.
Prices are daily reference observations; liquidity and execution checks still apply.


,yf_symbol,name,unavailable_reason
25,RQIH.L,R&Q Insurance,YFTzMissingError: $RQIH.L: possibly delisted; ...
29,FOG.L,Falcon Oil & Gas,YFTzMissingError: $FOG.L: possibly delisted; n...
69,SCE.L,Surface Transforms,YFTzMissingError: $SCE.L: possibly delisted; n...
73,UBG.L,Unbound Group,YFTzMissingError: $UBG.L: possibly delisted; n...
100,ZYT.L,Zytronic,YFTzMissingError: $ZYT.L: possibly delisted; n...
103,RNO.L,Renold,YFTzMissingError: $RNO.L: possibly delisted; n...
106,SIXH.L,600 Group,YFTzMissingError: $SIXH.L: possibly delisted; ...
141,AFRN.L,Aferian,YFTzMissingError: $AFRN.L: possibly delisted; ...
142,HRN.L,Hornby,YFTzMissingError: $HRN.L: possibly delisted; n...
144,FEN.L,Frenkel Topping,YFTzMissingError: $FEN.L: possibly delisted; n...


In [ ]:
import json

PERIODS = {"warmup": {"start": "2015-01-01", "end_exclusive": "2016-01-01"},
    "development": {"start": "2016-01-01", "end_exclusive": "2022-01-01"},
    "validation": {"start": "2022-01-01", "end_exclusive": "2024-01-01"},
    "holdout": {"start": "2024-01-01", "end_exclusive": "2026-01-01"}}

period_masks = pd.DataFrame({name: ((schedule.index >= bounds["start"])& (schedule.index < bounds["end_exclusive"]))
        for name, bounds in PERIODS.items()}, index=schedule.index)

#every session belongs to exactly one period
assert period_masks.sum(axis=1).eq(1).all()
assert period_masks.any().all()

protocol_path = PROJECT_DIR / "data" / "research_periods_v1.json"
if protocol_path.exists():
    if json.loads(protocol_path.read_text()) != PERIODS:
        raise ValueError("Research periods differ from the saved protocol. Review the change before replacing it.")
else:
    protocol_path.write_text(json.dumps(PERIODS, indent=2) + "\n")

period_summary = pd.DataFrame([{
            "period": name,
            "first_session": schedule.index[mask][0],
            "last_session": schedule.index[mask][-1],
            "sessions": int(mask.sum()),
        } for name, mask in period_masks.items()]).set_index("period")

display(period_summary)

,first_session,last_session,sessions
period,,,
warmup,2015-01-02,2015-12-31,253
development,2016-01-04,2021-12-31,1518
validation,2022-01-04,2023-12-29,501
holdout,2024-01-02,2025-12-31,507


In [ ]:
import momentum_features

importlib.reload(momentum_features)

LIQUIDITY = {
    "lookback_sessions": 60,
    "min_positive_volume_fraction": 0.95,
    "min_median_traded_value_gbp": 100_000.0,
}

liquidity = momentum_features.build_liquidity_features(
    prices,
    schedule,
    lookback=LIQUIDITY["lookback_sessions"],
)

liquidity["liquidity_eligible"] = (
    liquidity["liquidity_history_complete"]
    & liquidity["median_traded_value_gbp"].ge(
        LIQUIDITY["min_median_traded_value_gbp"]
    )
    & liquidity["positive_volume_fraction"].ge(
        LIQUIDITY["min_positive_volume_fraction"]
    )
    & prices["positive_reported_volume"].eq(True)
    & prices["close_reference_usable"].eq(True)
)

eligible_counts = (
    liquidity["liquidity_eligible"]
    .groupby(level="Date")
    .sum()
)

development_dates = schedule.index[period_masks["development"]]

display(
    eligible_counts.loc[development_dates]
    .describe()
    .to_frame("Stocks passing liquidity checks")
)

NameError: name 'prices' is not defined